# Week 8 Capstone — Meeting-to-Action Assistant

Self-contained capstone notebook for COSC 650. It converts an existing meeting transcript into an operational brief with an executive summary, decisions, action items and owners, blockers, and follow-up questions.

### Reuse map

| Technique | Reused from | Role |
|---|---:|---|
| Structured prompting/output | Week 3 | Stable JSON brief contract |
| Deterministic validation | Week 4 | Validate required fields |
| MiniLM + FAISS RAG | Week 5 | Retrieve task guidance |
| Cross-encoder reranking | Week 6 | Rerank retrieved context |
| Dynamic few-shot retrieval | Week 7 | Select demonstrations |

**Architecture:** transcript → MiniLM/FAISS retrieval → cross-encoder reranking → optional dynamic examples → Gemini Flash → deterministic validation → metrics/judge/failure trace.

A small local Qwen model is also measured on a narrow classification task; it is not forced into the final architecture.

In [23]:
# Uncomment once in a fresh Colab runtime if needed:
# %pip install -q sentence-transformers faiss-cpu openai pandas numpy tabulate transformers torch

import os,re,json,time,math,hashlib,random
import numpy as np,pandas as pd,faiss
from sentence_transformers import SentenceTransformer,CrossEncoder
from IPython.display import display,Markdown

try:
    from google.colab import userdata
    if not os.getenv('GEMINI_API_KEY'):
        k=userdata.get('GEMINI_API_KEY')
        if k: os.environ['GEMINI_API_KEY']=k
except Exception:
    pass

GEMINI_MODEL=os.getenv('GEMINI_MODEL','gemini-3.5-flash')
JUDGE_MODEL=os.getenv('GEMINI_JUDGE_MODEL',GEMINI_MODEL)
LOCAL_QWEN_MODEL=os.getenv('LOCAL_QWEN_MODEL','Qwen/Qwen2.5-0.5B-Instruct')
EMBED_MODEL='sentence-transformers/all-MiniLM-L6-v2'
RERANK_MODEL='cross-encoder/ms-marco-MiniLM-L-6-v2'
MAX_SYNTHESIS_CALLS,MAX_JUDGE_CALLS,MAX_CLASSIFIER_CALLS=30,10,1
PRELOADED_CACHE_JSON=''
CACHE=json.loads(PRELOADED_CACHE_JSON) if PRELOADED_CACHE_JSON.strip() else {'synthesis':{},'judge':{},'classifier':{}}
LIVE_SYNTHESIS=LIVE_JUDGE=LIVE_CLASSIFIER=0

def ckey(x): return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def client():
    from openai import OpenAI
    if not os.getenv('GEMINI_API_KEY'): raise RuntimeError('GEMINI_API_KEY is not set')
    return OpenAI(api_key=os.environ['GEMINI_API_KEY'],base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
def gemini(model,messages,temperature=0,max_attempts=6):
    delays=[2,4,8,16,32]
    for n in range(max_attempts):
        try: return client().chat.completions.create(model=model,messages=messages,temperature=temperature)
        except Exception as e:
            status=getattr(e,'status_code',None)
            if status not in {429,500,502,503,504} or n==max_attempts-1: raise
            wait=delays[min(n,len(delays)-1)]+random.random()
            print(f'Gemini temporary error {status}; retrying in {wait:.1f}s...')
            time.sleep(wait)
def parse_json(s):
    s=re.sub(r'^```(?:json)?\s*|\s*```$','',s.strip(),flags=re.I)
    return json.loads(s)
print('Gemini:',GEMINI_MODEL,'| Judge:',JUDGE_MODEL,'| Local:',LOCAL_QWEN_MODEL)

Gemini: gemini-3.5-flash | Judge: gemini-3.5-flash | Local: Qwen/Qwen2.5-0.5B-Instruct


## Held-out evaluation set

Ten cases are frozen before execution and kept separate from the few-shot pool.

In [24]:
CASES=[
{'id':'E01','t':"A: Maybe move launch to Monday. B: No, keep Friday. B: I'll send QA by 3.",'d':['Keep Friday launch.'],'a':[('Send QA by 3.','B')],'b':[],'q':[]},
{'id':'E02','t':"A: Export is missing IDs. B: I'll ask Finance to fix them.",'d':[],'a':[('Ask Finance to fix missing IDs.','B')],'b':['Export is missing IDs.'],'q':[]},
{'id':'E03','t':"A: Could we use vendor B? B: Legal has not reviewed it. A: Do not choose yet.",'d':['Do not choose a vendor yet.'],'a':[],'b':['Legal has not reviewed vendor B.'],'q':[]},
{'id':'E04','t':"A: Weekly dashboard might help. B: No cadence decision. B: I'll collect examples.",'d':[],'a':[('Collect dashboard examples.','B')],'b':[],'q':[]},
{'id':'E05','t':"A: Import failed on invalid dates. B: I'll fix dates today. A: Keep current mapping.",'d':['Keep current mapping.'],'a':[('Fix invalid dates today.','B')],'b':['Invalid dates caused import failure.'],'q':[]},
{'id':'E06','t':"A: Use Sep 1 through Sep 30 for both reports. B: I'll rerun them tomorrow.",'d':['Use Sep 1 through Sep 30.'],'a':[('Rerun reports tomorrow.','B')],'b':[],'q':[]},
{'id':'E07','t':"A: We could automate email. B: Not in v1; human review is required. A: I'll document the gate.",'d':['Human review is required in v1.'],'a':[('Document the approval gate.','A')],'b':[],'q':[]},
{'id':'E08','t':"A: Duplicate customers cause bad totals. B: I'll test customer ID as the key.",'d':[],'a':[('Test customer ID as matching key.','B')],'b':['Duplicate customers cause bad totals.'],'q':[]},
{'id':'E09','t':"A: Should we remove notes? B: Not yet. We agreed to keep the shorter form. A: I'll measure completion time.",'d':['Keep the shorter form.'],'a':[('Measure completion time.','A')],'b':[],'q':['Should notes be removed?']},
{'id':'E10','t':"A: SSO effort is unknown. B: Do not promise it yet. B: I'll ask Engineering. A: Do we need a separate coreference model?",'d':['Do not promise SSO yet.'],'a':[('Ask Engineering about SSO effort.','B')],'b':['SSO effort is unknown.'],'q':['Do we need a separate coreference model?']}
]
assert len(CASES)==10
display(pd.DataFrame([{'case_id':x['id'],'transcript':x['t']} for x in CASES]))

,case_id,transcript
0,E01,"A: Maybe move launch to Monday. B: No, keep Fr..."
1,E02,A: Export is missing IDs. B: I'll ask Finance ...
2,E03,A: Could we use vendor B? B: Legal has not rev...
3,E04,A: Weekly dashboard might help. B: No cadence ...
4,E05,A: Import failed on invalid dates. B: I'll fix...
5,E06,A: Use Sep 1 through Sep 30 for both reports. ...
6,E07,A: We could automate email. B: Not in v1; huma...
7,E08,A: Duplicate customers cause bad totals. B: I'...
8,E09,A: Should we remove notes? B: Not yet. We agre...
9,E10,A: SSO effort is unknown. B: Do not promise it...


## Reused retrieval and few-shot components

The RAG corpus contains **operational task guidance**, not external factual knowledge. The few-shot pool is separate from the held-out evaluation set.

This section deliberately displays:
1. the full RAG corpus,
2. the full few-shot pool,
3. a sample dense-retrieval + cross-encoder reranking result.

That makes the reused Week 5/6/7 components visible and inspectable in the executed notebook.

In [25]:
RULES=[
'Confirmed decisions require commitment, approval, agreement, selection, rejection, or another explicit commitment.',
'Suggestions, possibilities, brainstorms, and maybe/could/might statements are not decisions unless a later turn commits.',
'Action items are future work assigned or accepted by a person or team; never invent an owner.',
'A blocker is a concrete condition preventing, delaying, or risking progress.',
'Follow-up questions must still be unresolved at the end of the meeting.',
'All extracted operational items must be grounded in transcript evidence.'
]
SHOTS=[
('Maybe delay launch. No, keep Friday.','Decision: keep Friday; delay was only a suggestion.'),
('Could we use vendor B? Compare legal terms first.','No vendor decision was made.'),
("The export is missing IDs. I'll ask Finance to fix them.",'Action item: ask Finance to fix the IDs.'),
('Legal has not reviewed the terms.','Blocker: legal review is outstanding.'),
('Should we remove notes?','Unresolved follow-up question.')
]

display(Markdown('### RAG operational-guidance corpus'))
display(pd.DataFrame({'rule_id':[f'R{i+1}' for i in range(len(RULES))],'guidance':RULES}))
display(Markdown('### Dynamic few-shot demonstration pool'))
display(pd.DataFrame([{'example_id':f'F{i+1}','input':a,'target':b} for i,(a,b) in enumerate(SHOTS)]))

emb=SentenceTransformer(EMBED_MODEL)
reranker=CrossEncoder(RERANK_MODEL)
rv=emb.encode(RULES,normalize_embeddings=True).astype('float32'); ri=faiss.IndexFlatIP(rv.shape[1]); ri.add(rv)
sv=emb.encode([x[0] for x in SHOTS],normalize_embeddings=True).astype('float32'); si=faiss.IndexFlatIP(sv.shape[1]); si.add(sv)

def retrieve_scored(q,docs,index,text=lambda x:x,top_k=4):
    qv=emb.encode([q],normalize_embeddings=True).astype('float32')
    ds,ids=index.search(qv,min(6,len(docs)))
    cand=[{'item':docs[int(i)],'dense':float(s)} for s,i in zip(ds[0],ids[0]) if i>=0]
    rs=reranker.predict([(q,text(x['item'])) for x in cand])
    for x,s in zip(cand,rs): x['rerank']=float(s)
    return sorted(cand,key=lambda x:x['rerank'],reverse=True)[:top_k]
def rules_for(t): return [x['item'] for x in retrieve_scored(t,RULES,ri,top_k=4)]
def shots_for(t): return [x['item'] for x in retrieve_scored(t,SHOTS,si,text=lambda x:x[0],top_k=3)]

demo=CASES[0]['t']
rr=retrieve_scored(demo,RULES,ri,top_k=4)
ss=retrieve_scored(demo,SHOTS,si,text=lambda x:x[0],top_k=3)
display(Markdown('### Sample retrieval/reranking output'))
display(pd.DataFrame([{'rank':i+1,'rule':x['item'],'dense_score':round(x['dense'],4),'rerank_score':round(x['rerank'],4)} for i,x in enumerate(rr)]))
display(pd.DataFrame([{'rank':i+1,'example':x['item'][0],'target':x['item'][1],'dense_score':round(x['dense'],4),'rerank_score':round(x['rerank'],4)} for i,x in enumerate(ss)]))

### RAG operational-guidance corpus

,rule_id,guidance
0,R1,"Confirmed decisions require commitment, approv..."
1,R2,"Suggestions, possibilities, brainstorms, and m..."
2,R3,Action items are future work assigned or accep...
3,R4,"A blocker is a concrete condition preventing, ..."
4,R5,Follow-up questions must still be unresolved a...
5,R6,All extracted operational items must be ground...


### Dynamic few-shot demonstration pool

,example_id,input,target
0,F1,"Maybe delay launch. No, keep Friday.",Decision: keep Friday; delay was only a sugges...
1,F2,Could we use vendor B? Compare legal terms first.,No vendor decision was made.
2,F3,The export is missing IDs. I'll ask Finance to...,Action item: ask Finance to fix the IDs.
3,F4,Legal has not reviewed the terms.,Blocker: legal review is outstanding.
4,F5,Should we remove notes?,Unresolved follow-up question.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

### Sample retrieval/reranking output

,rank,rule,dense_score,rerank_score
0,1,Follow-up questions must still be unresolved a...,0.2915,-9.8041
1,2,All extracted operational items must be ground...,0.0693,-10.1442
2,3,"Suggestions, possibilities, brainstorms, and m...",0.3293,-10.1815
3,4,Action items are future work assigned or accep...,0.1031,-10.7557


,rank,example,target,dense_score,rerank_score
0,1,"Maybe delay launch. No, keep Friday.",Decision: keep Friday; delay was only a sugges...,0.5913,3.3585
1,2,Could we use vendor B? Compare legal terms first.,No vendor decision was made.,0.2373,-9.2964
2,3,Legal has not reviewed the terms.,Blocker: legal review is outstanding.,0.0241,-9.4868


## Structured extraction and deterministic validation

In [26]:
PROMPT='''Return JSON only with this exact shape:
{"executive_summary":"string","decisions":[{"decision":"string","evidence":"string"}],"action_items":[{"description":"string","owner":"string","evidence":"string"}],"blockers":[{"blocker":"string","evidence":"string"}],"follow_up_questions":[{"question":"string","evidence":"string"}]}
Report only confirmed decisions, not suggestions. Do not invent owners. Blockers must materially delay/risk progress. Questions must remain unresolved. Include transcript evidence.'''

def valid(b): return isinstance(b,dict) and all(k in b for k in ['executive_summary','decisions','action_items','blockers','follow_up_questions'])
def extract(t,cfg):
    global LIVE_SYNTHESIS
    k=ckey(['synth',GEMINI_MODEL,cfg,t])
    if k in CACHE['synthesis']: return CACHE['synthesis'][k]
    if LIVE_SYNTHESIS>=MAX_SYNTHESIS_CALLS: raise RuntimeError('synthesis cap reached')
    parts=[PROMPT]
    log={'rules':[],'examples':[]}
    if cfg in 'BC': log['rules']=rules_for(t); parts+=['Guidance:
'+'
'.join('- '+x for x in log['rules'])]
    if cfg=='C': log['examples']=shots_for(t); parts+=['Examples:
'+'
'.join(f'- {a} => {b}' for a,b in log['examples'])]
    st=time.perf_counter(); r=gemini(GEMINI_MODEL,[{'role':'user','content':'

'.join(parts)+'

Transcript:
'+t}]); latency=time.perf_counter()-st
    try: b=parse_json(r.choices[0].message.content)
    except Exception: b={}
    out={'brief':b,'valid':valid(b),'latency_s':latency,'retrieval':log}; CACHE['synthesis'][k]=out; LIVE_SYNTHESIS+=1; return out

STOP=set('the a an to of and or for in on by with is are be as at this that'.split())
def toks(s): return set(re.findall(r'[a-z0-9]+',str(s).lower()))-STOP
def sim(a,b):
    A,B=toks(a),toks(b); return len(A&B)/len(A|B) if A and B else 0.0
def match(pred,gold,thr=.30):
    used=set(); hits=[]
    for p in pred:
        opts=[(sim(p,g),i) for i,g in enumerate(gold) if i not in used]
        if opts:
            s,i=max(opts)
            if s>=thr: hits.append((p,i)); used.add(i)
    tp=len(hits); P=tp/len(pred) if pred else (1 if not gold else 0); R=tp/len(gold) if gold else (1 if not pred else 0); F=2*P*R/(P+R) if P+R else 0
    return P,R,F,hits
def score(c,r):
    b=r['brief'] if r.get('valid') else {}; gold=c
    d=[x.get('decision','') for x in b.get('decisions',[]) if isinstance(x,dict)]
    a=[x.get('description','') for x in b.get('action_items',[]) if isinstance(x,dict)]
    owners=[x.get('owner','') for x in b.get('action_items',[]) if isinstance(x,dict)]
    bl=[x.get('blocker','') for x in b.get('blockers',[]) if isinstance(x,dict)]
    q=[x.get('question','') for x in b.get('follow_up_questions',[]) if isinstance(x,dict)]
    dp,dr,df,_=match(d,gold['d']); ap,ar,af,pairs=match(a,[x[0] for x in gold['a']]); _,_,bf,_=match(bl,gold['b']); _,_,qf,_=match(q,gold['q'])
    owner_hits=[]
    for p,gi in pairs:
        pi=a.index(p); owner_hits.append(pi<len(owners) and owners[pi].lower()==gold['a'][gi][1].lower())
    oa=sum(owner_hits)/len(owner_hits) if owner_hits else (1 if not gold['a'] else 0)
    return {'case_id':c['id'],'decision_precision':dp,'decision_recall':dr,'decision_f1':df,'action_precision':ap,'action_recall':ar,'action_f1':af,'owner_accuracy':oa,'blocker_f1':bf,'question_f1':qf,'valid':float(r.get('valid',False)),'latency_s':r.get('latency_s',math.nan)}

SyntaxError: unterminated string literal (detected at line 13) (1101257330.py, line 13)

## A/B/C deterministic evaluation

A = transcript only; B = RAG guidance; C = RAG guidance + dynamic few-shot examples.

In [ ]:
RESULTS={}; rows=[]
if not os.getenv('GEMINI_API_KEY'):
    print('API-BLOCKED: add GEMINI_API_KEY.')
else:
    for cfg in 'ABC':
        RESULTS[cfg]={}
        for c in CASES:
            try: r=extract(c['t'],cfg)
            except Exception as e:
                print(cfg,c['id'],'failed:',e); r={'brief':{},'valid':False,'latency_s':math.nan,'retrieval':{'rules':[],'examples':[]}}
            RESULTS[cfg][c['id']]=r; s=score(c,r); s['config']=cfg; rows.append(s)
    METRICS=pd.DataFrame(rows)
    cols=['decision_precision','decision_recall','decision_f1','action_precision','action_recall','action_f1','owner_accuracy','blocker_f1','question_f1','valid','latency_s']
    SUMMARY=METRICS.groupby('config')[cols].mean().round(3)
    SUMMARY['extraction_score']=SUMMARY[['decision_f1','action_f1','owner_accuracy','blocker_f1','question_f1','valid']].mean(axis=1).round(3)
    SELECTED_CONFIG=SUMMARY.extraction_score.idxmax()
    display(METRICS.round(3)); display(SUMMARY); print('Selected configuration:',SELECTED_CONFIG)

## LLM-as-judge

The deterministically selected configuration is judged on all 10 held-out cases for correctness, grounding, and operational usefulness.

In [ ]:
JUDGE_DF=pd.DataFrame(); JUDGE_SUMMARY=pd.DataFrame()
def judge(c,b):
    global LIVE_JUDGE
    payload={'transcript':c['t'],'gold':{'decisions':c['d'],'actions':c['a'],'blockers':c['b'],'questions':c['q']},'candidate':b}
    k=ckey(['judge',JUDGE_MODEL,payload])
    if k in CACHE['judge']: return CACHE['judge'][k]
    if LIVE_JUDGE>=MAX_JUDGE_CALLS: raise RuntimeError('judge cap reached')
    p='Score 1-5 on correctness, grounding, operational_usefulness. Return JSON only with these keys and reason.'
    r=gemini(JUDGE_MODEL,[{'role':'user','content':p+'
'+json.dumps(payload)}]); out=parse_json(r.choices[0].message.content); CACHE['judge'][k]=out; LIVE_JUDGE+=1; return out
if RESULTS:
    jr=[]
    for c in CASES:
        try: jr.append({'case_id':c['id'],**judge(c,RESULTS[SELECTED_CONFIG][c['id']]['brief'])})
        except Exception as e: print('Judge',c['id'],'failed:',e)
    JUDGE_DF=pd.DataFrame(jr)
    if not JUDGE_DF.empty:
        JUDGE_SUMMARY=JUDGE_DF[['correctness','grounding','operational_usefulness']].mean().to_frame('average').round(3)
        display(JUDGE_DF); display(JUDGE_SUMMARY)

## Required failure mode and mitigation

In [ ]:
FAILURE=None; MITIGATION_BEFORE=MITIGATION_AFTER=None
if RESULTS:
    sc={cfg:{c['id']:score(c,RESULTS[cfg][c['id']]) for c in CASES} for cfg in 'ABC'}
    chosen=comp=None
    for c in CASES:
        if sc['C'][c['id']]['decision_f1']<sc['B'][c['id']]['decision_f1']: chosen,comp=c['id'],'dynamic few-shot context'; break
    if chosen is None:
        for c in CASES:
            if sc['B'][c['id']]['decision_f1']<sc['A'][c['id']]['decision_f1']: chosen,comp=c['id'],'retrieved/reranked context'; break
    if chosen is None:
        chosen=min((c['id'] for c in CASES),key=lambda x:sc[SELECTED_CONFIG][x]['decision_f1']); comp='generation/extraction stage'
    FAILURE={'case_id':chosen,'component':comp}; print('Failure case:',chosen,'| component:',comp)
    for cfg in 'ABC':
        print('
',cfg,json.dumps(RESULTS[cfg][chosen]['brief'],indent=2)); print('score:',sc[cfg][chosen]); print('retrieval:',RESULTS[cfg][chosen]['retrieval'])

HEDGE=['maybe','could','might','consider','possibly','what if']; COMMIT=['agreed','approved','keep','use','do not','required']
def mitigate(b):
    z=json.loads(json.dumps(b)); kept=[]
    for d in z.get('decisions',[]):
        e=d.get('evidence','').lower() if isinstance(d,dict) else str(d).lower()
        if any(h in e for h in HEDGE) and not any(k in e for k in COMMIT): continue
        kept.append(d)
    z['decisions']=kept; return z
if RESULTS:
    before=[]; after=[]
    for c in CASES:
        r=RESULTS[SELECTED_CONFIG][c['id']]; before.append(score(c,r)['decision_f1']); q=dict(r); q['brief']=mitigate(r['brief']); after.append(score(c,q)['decision_f1'])
    MITIGATION_BEFORE=round(np.mean(before),3); MITIGATION_AFTER=round(np.mean(after),3)
    print('Decision F1 before:',MITIGATION_BEFORE,'after:',MITIGATION_AFTER)

## Gemini vs small local Qwen

Both models classify the same 20 spans into decision, action_item, blocker, follow_up_question, or context. Qwen runs locally on CPU.

In [ ]:
CLS=[
('Maybe move launch to Monday.','context'),('No schedule change right now.','decision'),("I'll send QA by 3.",'action_item'),('SSO effort is unknown.','blocker'),('What is the SSO effort?','follow_up_question'),
('No cadence decision today.','context'),('Keep current mapping.','decision'),("I'll fix dates today.",'action_item'),('Legal has not reviewed terms.','blocker'),('Which agreement can Legal clear?','follow_up_question'),
('We agreed to keep shorter form.','decision'),("I'll measure completion time.",'action_item'),('Reports do not match due to date ranges.','blocker'),('Use Sep 1 through Sep 30.','decision'),('Could we automate email?','context'),
('Human review is required.','decision'),("I'll test customer ID as key.",'action_item'),('Is customer ID populated?','follow_up_question'),('Owner field is wrong.','blocker'),('Do we need a coreference model?','follow_up_question')]

def gemini_batch(items):
    global LIVE_CLASSIFIER
    k=ckey(['classifier',GEMINI_MODEL,[x[0] for x in items]])
    if k in CACHE['classifier']: return CACHE['classifier'][k]
    if LIVE_CLASSIFIER>=MAX_CLASSIFIER_CALLS: raise RuntimeError('classifier cap reached')
    p='Classify each numbered span into exactly one label: decision, action_item, blocker, follow_up_question, context. Return JSON only as {"labels":[...]}.
'+'
'.join(f'{i+1}. {t}' for i,(t,_) in enumerate(items))
    st=time.perf_counter(); r=gemini(GEMINI_MODEL,[{'role':'user','content':p}]); lat=time.perf_counter()-st; out={'labels':parse_json(r.choices[0].message.content)['labels'],'latency_s':lat}; CACHE['classifier'][k]=out; LIVE_CLASSIFIER+=1; return out
GEMINI_CLASSIFIER_DF=pd.DataFrame(); QWEN_DF=pd.DataFrame(); MODEL_COMPARISON=pd.DataFrame()
if os.getenv('GEMINI_API_KEY'):
    try:
        g=gemini_batch(CLS); per=g['latency_s']/len(CLS); GEMINI_CLASSIFIER_DF=pd.DataFrame([{'text':t,'gold':y,'pred':p,'correct':p==y,'latency_s':per} for (t,y),p in zip(CLS,g['labels'])]); display(GEMINI_CLASSIFIER_DF)
    except Exception as e: print('Gemini classifier failed:',e)
try:
    import torch
    from transformers import AutoTokenizer,AutoModelForCausalLM
    qt=AutoTokenizer.from_pretrained(LOCAL_QWEN_MODEL); qm=AutoModelForCausalLM.from_pretrained(LOCAL_QWEN_MODEL,torch_dtype='auto'); qm.eval()
    qrows=[]
    for t,y in CLS:
        msgs=[{'role':'system','content':'Classify into exactly one label: decision, action_item, blocker, follow_up_question, context. Return only the label.'},{'role':'user','content':t}]
        rendered=qt.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True); inp=qt(rendered,return_tensors='pt'); st=time.perf_counter()
        with torch.no_grad(): out=qm.generate(**inp,max_new_tokens=10,do_sample=False)
        lat=time.perf_counter()-st; pred=qt.decode(out[0][inp['input_ids'].shape[1]:],skip_special_tokens=True).strip().lower(); pred=re.sub(r'[^a-z_]','',pred); qrows.append({'text':t,'gold':y,'pred':pred,'correct':pred==y,'latency_s':lat})
    QWEN_DF=pd.DataFrame(qrows); display(QWEN_DF)
except Exception as e: print('Local Qwen failed:',e)
rows=[]
if not GEMINI_CLASSIFIER_DF.empty: rows.append({'model':'Gemini Flash','accuracy':GEMINI_CLASSIFIER_DF.correct.mean(),'mean_latency_s':GEMINI_CLASSIFIER_DF.latency_s.mean(),'cost':'free-tier API'})
if not QWEN_DF.empty: rows.append({'model':LOCAL_QWEN_MODEL,'accuracy':QWEN_DF.correct.mean(),'mean_latency_s':QWEN_DF.latency_s.mean(),'cost':'$0 local'})
MODEL_COMPARISON=pd.DataFrame(rows); display(MODEL_COMPARISON.round(3))

## Dynamic research note

In [ ]:
def make_note():
    if not RESULTS or JUDGE_DF.empty or FAILURE is None: display(Markdown('### Measured results

Complete the evaluation above first.')); return None
    det=SUMMARY.copy(); core=['decision_f1','action_f1','owner_accuracy','blocker_f1','question_f1','valid']; a=float(det.loc['A',core].mean()); b=float(det.loc['B',core].mean()); c=float(det.loc['C',core].mean())
    rag=f'B changed aggregate extraction from {a:.3f} to {b:.3f}.'; few=f'C changed aggregate extraction from {b:.3f} to {c:.3f}.'
    qtxt='Local-model comparison did not complete.'
    if len(MODEL_COMPARISON)>=2:
        g=float(MODEL_COMPARISON.iloc[0].accuracy); q=float(MODEL_COMPARISON.iloc[1].accuracy); qtxt=f'Qwen accuracy was {q:.3f} versus Gemini {g:.3f} on the 20-span narrow task.'
    note=f'''# Research Note — Meeting-to-Action Assistant

## Measured results

### A/B/C deterministic metrics

{det.to_markdown()}

### LLM-as-judge

Selected configuration: **{SELECTED_CONFIG}**

{JUDGE_SUMMARY.to_markdown()}

### Questions and answers

**Did retrieval help?** {rag}

**Did dynamic few-shot prompting help?** {few}

**What failure was identified?** Case **{FAILURE['case_id']}**, traced to **{FAILURE['component']}**.

**Did mitigation help?** Decision F1 changed from {MITIGATION_BEFORE} to {MITIGATION_AFTER}.

**How did local Qwen compare with Gemini?** {qtxt}

**Final architecture decision.** Keep the simplest configuration with the strongest held-out extraction score. Gemini remains responsible for full meeting interpretation and synthesis; retrieval, reranking, validation, metrics, and any local narrow classifier remain local/deterministic where measurements justify them.'''
    display(Markdown(note)); return note
RESEARCH_NOTE=make_note()

## Submission guard and cache export

In [ ]:
problems=[]
if not RESULTS or 'METRICS' not in globals() or len(METRICS)!=30: problems.append('A/B/C evaluation incomplete')
if JUDGE_DF.empty or len(JUDGE_DF)!=10: problems.append('10-case judge incomplete')
if FAILURE is None: problems.append('failure trace missing')
if MODEL_COMPARISON.empty or len(MODEL_COMPARISON)<2: problems.append('Gemini vs Qwen comparison incomplete')
if RESEARCH_NOTE is None: problems.append('dynamic research note missing')
print('READY FOR SUBMISSION' if not problems else 'NOT READY FOR SUBMISSION')
for p in problems: print('-',p)
CACHE_EXPORT=json.dumps(CACHE,separators=(',',':'),ensure_ascii=False)
print('
CACHE_EXPORT =',CACHE_EXPORT)
print('
HTML export: jupyter nbconvert --to html week8_meeting_to_action_capstone.ipynb')